# Comparative Study of Neural Network Regularization Techniques
## For Regression and Spam Classification

---

| Field | Details |
|---|---|
| **Author** | Sanman Kadam |
| **Project** | Neural Network Regularization Study |
| **Domain** | Deep Learning, Regularization, NLP |
| **Language** | Python 3.x |
| **Framework** | TensorFlow / Keras |
| **Date** | April 2026 |

---

## 1. Problem Statement

Deep neural networks possess a high degree of representational capacity, enabling them to model complex nonlinear relationships in data. However, this same capacity makes them prone to **overfitting** -- a condition where the model memorizes noise, outliers, and idiosyncrasies in the training data rather than learning the true underlying patterns. Overfitting leads to poor generalization, meaning the model performs well on training data but fails on unseen data.

In real-world applications, training data is often contaminated with measurement noise, labeling errors, and outliers. Without appropriate constraints on model complexity, neural networks will attempt to fit these artifacts, resulting in unstable predictions and inflated error metrics.

**Regularization** techniques address this problem by introducing constraints or penalties that discourage the model from becoming excessively complex. Different regularization methods operate through distinct mechanisms:

- **Weight penalties** (L1, L2) modify the loss function to penalize large weights
- **Stochastic methods** (Dropout) randomly disable neurons during training
- **Normalization methods** (Batch Normalization) stabilize internal activations
- **Data-level methods** (Shuffling) reduce ordering bias in training batches

The central question this study addresses is: **How do different regularization techniques compare in their ability to reduce overfitting and improve generalization across fundamentally different learning tasks?**

Specifically, this study investigates two contrasting scenarios:

1. A **regression task** where the data contains deliberately injected outliers, creating a high-noise environment where overfitting is severe and regularization is expected to provide significant benefit.
2. A **binary classification task** (spam detection) where TF-IDF features provide inherent sparsity and strong linear separability, creating an environment where regularization may have limited impact.

By comparing performance across both tasks, this study demonstrates that the effectiveness of regularization is not universal but depends critically on the data characteristics, feature representation, and task complexity.

## 2. Aim

To study and compare the effect of different regularization techniques (L1, L2, Dropout, Batch Normalization, and Data Shuffling) in neural networks for:

- A **regression problem** (synthetic cubic dataset with injected outliers)
- A **binary classification problem** (Spam detection using TF-IDF features)

## 3. Objectives

1. To understand overfitting in deep neural networks.
2. To apply L1, L2, Dropout, and Batch Normalization regularization.
3. To compare Mean Squared Error across regularization techniques (regression).
4. To compare accuracy and loss across regularization techniques (classification).
5. To analyze why regularization behaves differently in regression and text classification tasks.

## 4. Theoretical Background

| Technique | Mechanism | Effect |
|---|---|---|
| **L1 Regularization** | Adds absolute value of weights to the loss function | Promotes sparsity; drives some weights to zero |
| **L2 Regularization** | Adds squared value of weights to the loss function | Shrinks weights uniformly; prevents large weights |
| **Dropout** | Randomly deactivates neurons during training | Prevents co-adaptation; acts as ensemble |
| **Batch Normalization** | Normalizes layer inputs to zero mean and unit variance | Stabilizes training; reduces internal covariate shift |
| **Data Shuffling** | Randomizes sample order each epoch | Reduces sequence bias; improves gradient estimates |

## 5. Environment Setup and Library Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import regularizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print("Environment ready.")

---
# PART A: Regression Problem
## Synthetic Cubic Dataset with Outlier Injection
---

## 6. Data Generation

A synthetic dataset is generated from the cubic function `f(x) = x^3 + 2x^2 - x` with added Gaussian noise. This simulates a real-world regression scenario where the underlying relationship is nonlinear and observations contain measurement error.

In [ ]:
def generate_data(seed=43, std=0.1, samples=500):
    """Generate synthetic cubic data with Gaussian noise."""
    np.random.seed(seed)
    X = np.linspace(-1, 1, samples)
    f = X**3 + 2*X**2 - X
    y = f + np.random.randn(samples) * std
    return X, y

X, y = generate_data()
f = X**3 + 2*X**2 - X

plt.figure(figsize=(10, 5))
plt.plot(X, y, 'rx', alpha=0.5, label='Data Samples')
plt.plot(X, f, 'b-', linewidth=2, label='True Function')
plt.title('Synthetic Cubic Data with Gaussian Noise', fontsize=14)
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Interpretation

The plot displays red data points scattered around a smooth cubic curve. The noise standard deviation of 0.1 introduces minor perturbations around the true function. This represents a clean regression scenario where the underlying signal is recoverable.

## 7. Outlier Injection

To stress-test regularization methods, extreme outliers are injected at six positions across the dataset. These outliers simulate corrupted measurements or anomalous observations commonly encountered in practice.

In [ ]:
# Inject outliers at specific positions
y[20:30] = 0
y[100:110] = 2
y[180:190] = 4
y[260:270] = -2
y[340:350] = -3
y[420:430] = 4

plt.figure(figsize=(10, 5))
plt.plot(X, y, 'rx', alpha=0.5, label='Data Samples (with Outliers)')
plt.plot(X, f, 'b-', linewidth=2, label='True Function')
plt.title('Cubic Data with Injected Outliers', fontsize=14)
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Interpretation

The updated plot reveals clusters of points displaced far from the true cubic curve. These extreme values (ranging from -3 to 4) create a challenging learning environment. A model that overfits will attempt to pass through these outliers, distorting its learned representation of the underlying function.

## 8. Baseline Model (No Regularization)

In [ ]:
model = Sequential([
    Dense(1000, activation='relu', input_shape=(1,)),
    Dense(120, activation='relu'),
    Dense(120, activation='relu'),
    Dense(1)
])

model.compile(optimizer=Adam(learning_rate=1e-3), loss='mean_squared_error')
print(model.summary())
model.fit(X, y, epochs=20, batch_size=100, verbose=1)

In [ ]:
y_pred_noreg = model.predict(X)

plt.figure(figsize=(10, 5))
plt.plot(X, y, 'rx', alpha=0.5, label='Data Samples')
plt.plot(X, f, 'b-', linewidth=2, label='True Function')
plt.plot(X, y_pred_noreg, 'g-', linewidth=2, label='Predicted (No Regularization)')
plt.title('Baseline Model -- No Regularization', fontsize=14)
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

no_reg = np.mean((y - y_pred_noreg)**2)
print(f"Mean Squared Error (No Regularization): {no_reg:.4f}")

### Interpretation

The training loss decreases rapidly across epochs, indicating the model is memorizing the training data effectively. However, the prediction curve exhibits sharp bends near outlier positions rather than following the smooth cubic trend. The MSE is comparatively high (approximately 1.92), confirming that the model is overfitting -- it captures noise and outliers rather than the true underlying pattern. The high model complexity (1000 neurons in the first layer, 120 in subsequent layers) combined with the absence of any regularization makes this model highly susceptible to overfitting.

## 9. L1 Regularization (Lasso)

In [ ]:
model_l1 = Sequential([
    Dense(1000, activation='relu', input_shape=(1,),
          kernel_regularizer=keras.regularizers.l1(l1=0.01)),
    Dense(120, activation='relu',
          kernel_regularizer=keras.regularizers.l1(l1=0.001)),
    Dense(120, activation='relu'),
    Dense(1)
])

model_l1.compile(optimizer=Adam(learning_rate=1e-3), loss='mean_squared_error')
model_l1.fit(X, y, epochs=20, batch_size=100, verbose=1)

In [ ]:
y_pred_l1 = model_l1.predict(X)

plt.figure(figsize=(10, 5))
plt.plot(X, y, 'rx', alpha=0.5, label='Data Samples')
plt.plot(X, f, 'b-', linewidth=2, label='True Function')
plt.plot(X, y_pred_l1, 'g-', linewidth=2, label='Predicted (L1)')
plt.title('L1 Regularization (Lasso)', fontsize=14)
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

l1_mse = np.mean((y - y_pred_l1)**2)
print(f"Mean Squared Error (L1): {l1_mse:.4f}")

### Interpretation

With L1 regularization applied, the loss decreases in a controlled manner during training. The predicted curve is noticeably smoother compared to the unregularized baseline and does not aggressively fit the outlier points. The MSE decreases to approximately 1.68, representing an improvement over the baseline.

L1 regularization penalizes the absolute magnitude of weights, driving some weights exactly to zero. This effectively performs feature selection within the network, reducing model complexity and producing a more parsimonious representation. The resulting model prioritizes the dense central data region and demonstrates improved robustness against extreme outliers.

## 10. L2 Regularization (Ridge)

In [ ]:
model_l2 = Sequential([
    Dense(1000, activation='relu', input_shape=(1,),
          kernel_regularizer=keras.regularizers.l2(l2=0.0001)),
    Dense(120, activation='relu',
          kernel_regularizer=keras.regularizers.l2(l2=0.0001)),
    Dense(120, activation='relu',
          kernel_regularizer=keras.regularizers.l2(l2=0.0001)),
    Dense(1)
])

model_l2.compile(optimizer=Adam(learning_rate=1e-3), loss='mean_squared_error')
model_l2.fit(X, y, validation_split=0.2, epochs=20, batch_size=40, verbose=1)

In [ ]:
y_pred_l2 = model_l2.predict(X)

plt.figure(figsize=(10, 5))
plt.plot(X, y, 'rx', alpha=0.5, label='Data Samples')
plt.plot(X, f, 'b-', linewidth=2, label='True Function')
plt.plot(X, y_pred_l2, 'g-', linewidth=2, label='Predicted (L2)')
plt.title('L2 Regularization (Ridge)', fontsize=14)
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

l2_mse = np.mean((y - y_pred_l2)**2)
print(f"Mean Squared Error (L2): {l2_mse:.4f}")

### Interpretation

L2 regularization produces stable training and validation loss curves throughout the 20 epochs. The predicted curve closely follows the true cubic function and does not overreact to noisy data points or outliers.

Unlike L1, which drives weights to exactly zero, L2 regularization shrinks all weights uniformly toward zero without eliminating them. This preserves all features while reducing the magnitude of each weight, resulting in a smoother decision boundary. The combined effect is reduced variance in predictions and improved generalization to unseen data.

## 11. Dropout Regularization

In [ ]:
model_dp = Sequential([
    Dense(1000, activation='relu', input_shape=(1,)),
    Dropout(0.1),
    Dense(120, activation='relu'),
    Dropout(0.1),
    Dense(120, activation='relu'),
    Dropout(0.1),
    Dense(1)
])

model_dp.compile(optimizer=Adam(learning_rate=1e-3), loss='mean_squared_error')
model_dp.fit(X, y, validation_split=0.2, epochs=20, batch_size=40, verbose=1)

In [ ]:
y_pred_dp = model_dp.predict(X)

plt.figure(figsize=(10, 5))
plt.plot(X, y, 'rx', alpha=0.5, label='Data Samples')
plt.plot(X, f, 'b-', linewidth=2, label='True Function')
plt.plot(X, y_pred_dp, 'g-', linewidth=2, label='Predicted (Dropout)')
plt.title('Dropout Regularization (rate=0.1)', fontsize=14)
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

dp_mse = np.mean((y - y_pred_dp)**2)
print(f"Mean Squared Error (Dropout): {dp_mse:.4f}")

### Interpretation

With a dropout rate of 0.1, 10% of neurons are randomly deactivated during each training step. The predicted curve appears smoother than the baseline and does not tightly follow noisy data points.

Dropout prevents co-adaptation of neurons by forcing the network to learn redundant representations. During inference, all neurons are active but their outputs are scaled, effectively averaging predictions from an ensemble of sub-networks. This mechanism reduces overfitting and improves robustness without explicitly modifying the loss function.

## 12. Batch Normalization

In [ ]:
model_bn = Sequential([
    Dense(1000, activation='relu', input_shape=(1,)),
    BatchNormalization(),
    Dense(120, activation='relu'),
    Dense(120, activation='relu'),
    Dense(1)
])

model_bn.compile(optimizer=Adam(learning_rate=1e-3), loss='mean_squared_error')
model_bn.fit(X, y, validation_split=0.2, epochs=20, batch_size=40, verbose=1)

In [ ]:
y_pred_bn = model_bn.predict(X)

plt.figure(figsize=(10, 5))
plt.plot(X, y, 'rx', alpha=0.5, label='Data Samples')
plt.plot(X, f, 'b-', linewidth=2, label='True Function')
plt.plot(X, y_pred_bn, 'g-', linewidth=2, label='Predicted (BatchNorm)')
plt.title('Batch Normalization', fontsize=14)
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

bn_mse = np.mean((y - y_pred_bn)**2)
print(f"Mean Squared Error (BatchNorm): {bn_mse:.4f}")

### Interpretation

Batch Normalization normalizes the inputs to each layer by adjusting and scaling activations to have zero mean and unit variance. This reduces internal covariate shift -- the phenomenon where the distribution of layer inputs changes during training.

The resulting prediction curve is smooth with less extreme bending near outlier regions. Training is more stable, and the MSE shows moderate improvement. While BatchNorm is primarily designed to accelerate training convergence, it provides a mild regularization effect through the noise introduced by mini-batch statistics.

## 13. Data Shuffling

In [ ]:
model_sh = Sequential([
    Dense(1000, activation='relu', input_shape=(1,)),
    Dense(120, activation='relu'),
    Dense(120, activation='relu'),
    Dense(1)
])

model_sh.compile(optimizer=Adam(learning_rate=1e-3), loss='mean_squared_error')
model_sh.fit(X, y, validation_split=0.2, epochs=20, batch_size=40, shuffle=True, verbose=1)

In [ ]:
y_pred_sh = model_sh.predict(X)

plt.figure(figsize=(10, 5))
plt.plot(X, y, 'rx', alpha=0.5, label='Data Samples')
plt.plot(X, f, 'b-', linewidth=2, label='True Function')
plt.plot(X, y_pred_sh, 'g-', linewidth=2, label='Predicted (Shuffled)')
plt.title('Data Shuffling', fontsize=14)
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

sh_mse = np.mean((y - y_pred_sh)**2)
print(f"Mean Squared Error (Shuffled): {sh_mse:.4f}")

### Interpretation

Data shuffling randomizes the order of training samples at each epoch. This prevents the model from learning sequential patterns in the data ordering and produces more representative mini-batch gradient estimates.

The predicted curve is slightly smoother than the baseline, but the improvement is marginal compared to L1 and L2 regularization. Shuffling is a lightweight technique that provides a minor regularization benefit and is generally recommended as a default training practice rather than a standalone regularization strategy.

## 14. Regression Results -- Comparative Summary

In [ ]:
names = ['No Reg.', 'L1', 'L2', 'Dropout', 'BatchNorm', 'Shuffling']
errors = [no_reg, l1_mse, l2_mse, dp_mse, bn_mse, sh_mse]

# Bar chart comparison
plt.figure(figsize=(12, 6))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12', '#1abc9c']
bars = plt.bar(names, errors, width=0.6, color=colors, edgecolor='black', linewidth=0.8)

for bar, err in zip(bars, errors):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
             f'{err:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.title('Mean Squared Error Comparison Across Regularization Techniques', fontsize=14)
plt.ylabel('Mean Squared Error', fontsize=12)
plt.xlabel('Regularization Technique', fontsize=12)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Summary table
results_df = pd.DataFrame({
    'Technique': names,
    'MSE': [f'{e:.4f}' for e in errors]
})
print('\nRegression Results Summary')
print('=' * 40)
print(results_df.to_string(index=False))

### Interpretation

The bar chart provides a direct visual comparison of MSE values across all regularization techniques. Key observations:

1. The **baseline model** (no regularization) exhibits the highest MSE, confirming severe overfitting to outlier-contaminated data.
2. **L1 regularization** achieves a meaningful reduction in MSE by promoting weight sparsity and effectively ignoring irrelevant features.
3. **L2 regularization** provides stable performance by uniformly shrinking weights, preventing any single weight from dominating.
4. **Dropout** reduces overfitting through stochastic neuron deactivation, yielding moderate improvement.
5. **Batch Normalization** offers training stability with a mild regularization side-effect.
6. **Data Shuffling** provides the least improvement, serving more as a training best practice than a regularization technique.

Overall, explicit regularization techniques (L1, L2, Dropout) consistently outperform implicit methods (BatchNorm, Shuffling) for regression tasks with noisy, outlier-heavy data.

---
# PART B: Classification Problem
## Spam Detection Using TF-IDF Features
---

## 15. Data Loading and Preprocessing

In [ ]:
data = pd.read_csv('Data/spam.csv', encoding='latin-1')

# Keep only required columns
data = data[['v1', 'v2']]
data.columns = ['label', 'text']
data['label'] = data['label'].map({'ham': 0, 'spam': 1})

print('Dataset Shape:', data.shape)
print('\nClass Distribution:')
print(data['label'].value_counts().rename({0: 'Ham', 1: 'Spam'}))
print('\nSample Records:')
data.head()

In [ ]:
sw = stopwords.words('english')
vectorizer = TfidfVectorizer(stop_words=sw)

X_cls = vectorizer.fit_transform(data['text']).toarray()
y_cls = data['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42
)

input_dim = X_cls.shape[1]
print(f'Feature Dimensions (TF-IDF): {input_dim}')
print(f'Training Samples: {X_train.shape[0]}')
print(f'Testing Samples: {X_test.shape[0]}')

### Interpretation

The spam dataset contains SMS messages labeled as ham (legitimate) or spam. After TF-IDF vectorization with English stopword removal, each message is represented as a high-dimensional sparse vector. The 80-20 train-test split ensures sufficient data for both training and evaluation. The class distribution is imbalanced, with ham messages significantly outnumbering spam.

## 16. Utility Functions

In [ ]:
def plot_classification_metrics(history):
    """Plot training and validation accuracy and loss curves."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Accuracy
    axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
    axes[0].set_title('Accuracy', fontsize=13)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Loss
    axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
    axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    axes[1].set_title('Loss', fontsize=13)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def build_and_evaluate(reg=None, epochs=10):
    """Build, train, and evaluate a classification model with specified regularization."""
    model = Sequential()
    model.add(Dense(512, activation='relu', input_shape=(input_dim,)))

    if reg == 'L1':
        model.add(Dense(256, activation='relu',
                        kernel_regularizer=regularizers.l1(0.001)))
        model.add(Dense(64, activation='relu',
                        kernel_regularizer=regularizers.l1(0.001)))
    elif reg == 'L2':
        model.add(Dense(256, activation='relu',
                        kernel_regularizer=regularizers.l2(0.001)))
        model.add(Dense(64, activation='relu',
                        kernel_regularizer=regularizers.l2(0.001)))
    elif reg == 'Dropout':
        model.add(Dropout(0.3))
        model.add(Dense(256, activation='relu'))
        model.add(Dropout(0.3))
        model.add(Dense(64, activation='relu'))
    elif reg == 'BatchNorm':
        model.add(BatchNormalization())
        model.add(Dense(256, activation='relu'))
        model.add(BatchNormalization())
        model.add(Dense(64, activation='relu'))
    else:
        model.add(Dense(256, activation='relu'))
        model.add(Dense(64, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

    history = model.fit(X_train, y_train,
                        validation_data=(X_test, y_test),
                        epochs=epochs, batch_size=64, verbose=1)

    plot_classification_metrics(history)

    y_pred = (model.predict(X_test) > 0.5).astype('int32')
    print('\nClassification Report:')
    print(classification_report(y_test, y_pred))
    print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')

    return model, history

## 17. Classification -- Baseline Model (No Regularization)

In [ ]:
print('Baseline Model (No Regularization)')
print('=' * 50)
base_model, base_hist = build_and_evaluate()

### Interpretation

The baseline model achieves high training accuracy rapidly, indicating sufficient model capacity. The slight gap between training and validation accuracy suggests mild overfitting. However, the overall validation accuracy remains high because the TF-IDF representation of spam data provides strong linear separability -- the feature space naturally separates ham from spam without requiring complex decision boundaries.

## 18. Classification -- L1 Regularization

In [ ]:
print('L1 Regularization')
print('=' * 50)
model_18, hist_18 = build_and_evaluate(reg='L1')

### Interpretation

L1 regularization does not significantly improve classification performance over the baseline. This is because TF-IDF features are already inherently sparse -- most words appear in very few documents, resulting in vectors dominated by zeros. Since L1 promotes sparsity in weights, applying it to an already sparse feature space provides diminishing returns. The model performance remains comparable to the baseline.

## 19. Classification -- L2 Regularization

In [ ]:
print('L2 Regularization')
print('=' * 50)
model_19, hist_19 = build_and_evaluate(reg='L2')

### Interpretation

L2 regularization improves weight stability by preventing any individual weight from growing disproportionately large. However, it does not drastically change classification accuracy. The spam dataset's inherent linear separability means that even without regularization, the model can achieve strong performance. L2 provides marginal improvements in validation stability but the overall accuracy difference is negligible.

## 20. Classification -- Dropout

In [ ]:
print('Dropout')
print('=' * 50)
model_20, hist_20 = build_and_evaluate(reg='Dropout')

### Interpretation

Dropout with a rate of 0.3 shows minimal improvement over the baseline. The strong linear separability of the TF-IDF spam features means the classification task does not require complex neuron interactions. Dropout's strength lies in preventing co-adaptation in models that learn intricate nonlinear patterns -- a scenario that does not strongly apply to this dataset.

## 21. Classification -- Batch Normalization

In [ ]:
print('Batch Normalization')
print('=' * 50)
model_21, hist_21 = build_and_evaluate(reg='BatchNorm')

### Interpretation

Batch Normalization stabilizes the learning process and may accelerate convergence, but it does not outperform the baseline significantly in terms of final accuracy. The normalization of layer inputs reduces sensitivity to weight initialization and learning rate choices, producing more consistent training curves. However, for this particular dataset, the accuracy ceiling is already approached by the baseline model.

---
## 22. Overall Conclusion

### Regression Task

Regularization techniques demonstrate clear and measurable benefits for the regression task with outlier-contaminated data:

| Observation | Detail |
|---|---|
| Best Performer | L1 Regularization (lowest MSE, highest outlier robustness) |
| Runner-up | L2 Regularization (smooth weight shrinkage, stable predictions) |
| Moderate Effect | Dropout (stochastic regularization, moderate MSE reduction) |
| Mild Effect | Batch Normalization (training stability, mild regularization) |
| Minimal Effect | Data Shuffling (minor improvement, best used as default practice) |

### Classification Task

Regularization techniques do not produce significant accuracy improvements for spam classification because:

1. **TF-IDF features provide inherent sparsity**, reducing the need for L1 weight pruning.
2. **The classification boundary is approximately linear**, meaning the baseline model does not severely overfit.
3. **The dataset is relatively clean** without the injected noise and outliers present in the regression task.

### Key Takeaway

The effectiveness of regularization is task-dependent. For complex, noisy regression problems, regularization is essential. For classification tasks with well-separated features, regularization provides marginal gains and may even slightly hinder performance by unnecessarily constraining model capacity.

---

| Field | Details |
|---|---|
| **Author** | Sanman Kadam |
| **Project** | Comparative Study of Neural Network Regularization Techniques |
| **Status** | Complete |